## TableMate: Cell 1 - Projekt och importer
TableMate är ett terminalbaserat bokningssystem där restauranggäster kan söka efter lediga bord, få en rekommendation, välja ett specifikt bord och avboka en bokning med ett boknings-ID.

Programmet använder standardbiblioteken `csv`, `os`, `datetime` och `timedelta`. Det externa biblioteket `openai` används för att kommunicera med xAI:s Grok API.

Konstanter används för filnamn, bokningstider, modellnamn och kolumnerna i bokningsfilen.

In [ ]:
import csv # Importerar Pythons standardbibliotek för CSV-filer 
import os # hämtar API-nyckeln från datorns miljövariabler.
from datetime import datetime, timedelta # Importerar klasser för datum, klockslag och beräkning av framtida datum.
from openai import ( # Importerar felklasser och API-klienten från det externa biblioteket openai.
    APIConnectionError, # Fel när programmet inte kan nå API-tjänsten.
    APIError, # Ett mer allmänt fel från API-tjänsten.
    AuthenticationError, # Fel när API-nyckeln är ogiltig.
    OpenAI, # Klienten som används för att skicka API-anropet.
    RateLimitError, # Fel när för många anrop har skickats.
)


TABLES_FILE = "tables.csv" 
BOOKINGS_FILE = "bookings.csv"
GROK_MODEL = "grok-4.7"
TIME_SLOTS = ["17:00", "18:00", "19:00", "20:00", "21:00"]

BOOKING_FIELDS = [
    "booking_id",
    "customer_name",
    "booking_date",
    "booking_time",
    "guest_count",
    "table_id",
    "booking_status",
    "created_at",
]

## Cell 2 - Klasser och arv

Klassen `Table` representerar ett restaurangbord. Varje objekt innehåller bords-ID, kapacitet, placering, typ, beskrivning och information om bordet är bokningsbart.

Metoden `display_information()` returnerar information om bordet. Metoden `can_seat_guests()` kontrollerar om bordet är bokningsbart och har tillräckligt många platser.

`TerraceTable` ärver från `Table`. Jag valde arv eftersom ett terrassbord är en särskild typ av bord. Barnklassen återanvänder basklassens funktionalitet men anpassar informationen för utomhusbord.

In [ ]:
class Table:
    def __init__(
        self,
        table_id,
        table_capacity,
        table_placement,
        table_type,
        table_description,
        is_bookable,
    ):
        self.table_id = table_id
        self.table_capacity = table_capacity
        self.table_placement = table_placement
        self.table_type = table_type
        self.table_description = table_description
        self.is_bookable = is_bookable

    def display_information(self):
        return (
            f"Table: {self.table_id}, "
            f"Seats: {self.table_capacity}, "
            f"Placement: {self.table_placement}, "
            f"Type: {self.table_type}, "
            f"Description: {self.table_description}, "
            f"Bookable: {self.is_bookable}"
        )

    def can_seat_guests(self, guest_count):
        return self.is_bookable and guest_count <= self.table_capacity


class TerraceTable(Table):
    def display_information(self):
        return f"{super().display_information()}, Outdoor seating: Yes"

## Cell 3 - Läsa data från CSV

`text_to_bool()` omvandlar textvärden som `yes`, `true` och `1` till det booleska värdet `True`.

`load_tables()` läser borden från `tables.csv` och skapar objekt av `Table` eller `TerraceTable`.

`load_bookings()` läser tidigare bokningar från `bookings.csv`. Eftersom CSV-data först läses som text omvandlas exempelvis bords-ID, boknings-ID och antal gäster till heltal.

`try/except` används för att hantera saknade filer, felaktiga värden, saknade kolumner och CSV-fel.

In [ ]:
def text_to_bool(value):
    return value.strip().lower() in ("true", "yes", "1")


def load_tables(filename):
    tables = []

    try:
        with open(filename, "r", encoding="utf-8", newline="") as file:
            for row in csv.DictReader(file):
                table_class = (
                    TerraceTable
                    if row["table_type"].strip().lower() == "terrace"
                    else Table
                )

                table = table_class(
                    int(row["table_id"]),
                    int(row["table_capacity"]),
                    row["table_placement"].strip(),
                    row["table_type"].strip(),
                    row["table_description"].strip(),
                    text_to_bool(row["is_bookable"]),
                )
                tables.append(table)

    except FileNotFoundError:
        print(f"Error: '{filename}' was not found.")
    except (KeyError, ValueError, csv.Error, OSError) as error:
        print(f"Error while reading '{filename}': {error}")

    return tables


def load_bookings(filename):
    bookings = []

    try:
        with open(filename, "r", encoding="utf-8", newline="") as file:
            for row in csv.DictReader(file):
                row["booking_id"] = int(row["booking_id"])
                row["guest_count"] = int(row["guest_count"])
                row["table_id"] = int(row["table_id"])
                bookings.append(row)

    except FileNotFoundError:
        print(f"'{filename}' does not exist yet. A new file will be created.")
    except (KeyError, ValueError, csv.Error, OSError) as error:
        print(f"Error while reading '{filename}': {error}")

    return bookings


## Cell 4 - Spara bokningar

`save_bookings()` sparar den aktuella listan med bokningar i `bookings.csv`.

Funktionen använder `csv.DictWriter` för att skriva kolumnrubriker och bokningsrader. Den returnerar `True` om sparningen lyckas och `False` om ett fel inträffar.

Returvärdet gör att andra funktioner kan kontrollera om ändringen verkligen sparades.

In [ ]:
def save_bookings(filename, bookings):
    try:
        with open(filename, "w", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=BOOKING_FIELDS)
            writer.writeheader()
            writer.writerows(bookings)
        return True
    except (OSError, csv.Error) as error:
        print(f"Error while saving bookings: {error}")
        return False


## Cell 5 - Tillgänglighet, bokning och avbokning

`is_table_available()` kontrollerar om ett bord redan har en aktiv bokning på samma datum och tid.

`find_available_tables()` filtrerar fram bord som är bokningsbara, har tillräcklig kapacitet och är lediga.

`create_booking()` skapar ett nytt unikt boknings-ID, lägger till bokningen och sparar den i CSV-filen.

`cancel_booking()` söker efter en aktiv bokning med rätt ID och ändrar dess status till `cancelled`. Bokningen raderas inte eftersom bokningshistoriken ska finnas kvar.

In [ ]:
def is_table_available(bookings, table_id, booking_date, booking_time):
    for booking in bookings:
        if (
            booking["table_id"] == table_id
            and booking["booking_date"] == booking_date
            and booking["booking_time"] == booking_time
            and booking["booking_status"].strip().lower() == "active"
        ):
            return False
    return True


def find_available_tables(
    tables, bookings, guest_count, booking_date, booking_time
):
    return [
        table
        for table in tables
        if table.can_seat_guests(guest_count)
        and is_table_available(
            bookings, table.table_id, booking_date, booking_time
        )
    ]


def create_booking(
    bookings,
    table,
    customer_name,
    booking_date,
    booking_time,
    guest_count,
    filename,
):
    if not table.can_seat_guests(guest_count):
        print("The table cannot accommodate this group.")
        return None

    if not is_table_available(
        bookings, table.table_id, booking_date, booking_time
    ):
        print("The table is already booked at this date and time.")
        return None

    new_booking = {
        "booking_id": max(
            (booking["booking_id"] for booking in bookings), default=0
        )
        + 1,
        "customer_name": customer_name,
        "booking_date": booking_date,
        "booking_time": booking_time,
        "guest_count": guest_count,
        "table_id": table.table_id,
        "booking_status": "active",
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    bookings.append(new_booking)

    if save_bookings(filename, bookings):
        return new_booking

    bookings.pop()
    return None


def cancel_booking(bookings, booking_id, filename):
    booking = next(
        (
            item
            for item in bookings
            if item["booking_id"] == booking_id
            and item["booking_status"].strip().lower() == "active"
        ),
        None,
    )

    if booking is None:
        print(f"No active booking with ID {booking_id} was found.")
        return False

    booking["booking_status"] = "cancelled"

    if save_bookings(filename, bookings):
        print(f"Booking {booking_id} was cancelled.")
        return True

    booking["booking_status"] = "active"
    return False


## Cell 6 - Lokal rekommendation och Grok API

`recommend_table_locally()` jämför kundens önskemål med de lediga bordens placering, typ och beskrivning.

`recommend_table_with_grok()` försöker hämta en rekommendation från Grok. Endast bord som Python redan har kontrollerat som lediga skickas till AI-tjänsten.

API-nyckeln läses från miljövariabeln `XAI_API_KEY` och sparas därför inte i koden.

Om API-nyckeln saknas eller om ett API-fel inträffar används den lokala rekommendationen som reservlösning. Kundens namn skickas inte till AI-tjänsten.

In [ ]:
def recommend_table_locally(available_tables, customer_preference):
    ignored_words = {
        "bord",
        "table",
        "plats",
        "place",
        "want",
        "would",
        "like",
        "sitta",
        "gärna",
    }
    keywords = {
        word.strip(".,!?")
        for word in customer_preference.lower().split()
        if len(word.strip(".,!?")) >= 4
        and word.strip(".,!?") not in ignored_words
    }

    matching_tables = []
    for table in available_tables:
        searchable_text = " ".join(
            [
                table.table_placement,
                table.table_type,
                table.table_description,
            ]
        ).lower()

        if any(keyword in searchable_text for keyword in keywords):
            matching_tables.append(table)

    recommended = min(
        matching_tables or available_tables,
        key=lambda table: (table.table_capacity, table.table_id),
    )
    return (
        f"Recommended table {recommended.table_id}: "
        f"{recommended.table_description}."
    )


def recommend_table_with_grok(available_tables, customer_preference):
    fallback = recommend_table_locally(
        available_tables, customer_preference
    )
    api_key = os.getenv("XAI_API_KEY")

    if not api_key:
        print("Grok is unavailable. Using a local recommendation.")
        return fallback

    table_information = "\n".join(
        f"Table {table.table_id}: {table.display_information()}"
        for table in available_tables
    )

    try:
        client = OpenAI(
            api_key=api_key,
            base_url="https://api.x.ai/v1",
            timeout=20.0,
        )
        response = client.responses.create(
            model=GROK_MODEL,
            store=False,
            input=[
                {
                    "role": "system",
                    "content": (
                        "You are a restaurant booking assistant. "
                        "Recommend exactly one table from the provided list. "
                        "Give the table ID and a short reason."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        f"Customer preference: {customer_preference}\n\n"
                        f"Available tables:\n{table_information}"
                    ),
                },
            ],
        )
        return response.output_text.strip()

    except AuthenticationError:
        print("The xAI API key is invalid. Using a local recommendation.")
    except RateLimitError:
        print("The xAI rate limit was reached. Using a local recommendation.")
    except APIConnectionError:
        print("Could not connect to xAI. Using a local recommendation.")
    except APIError as error:
        print(f"xAI API error: {error}. Using a local recommendation.")

    return fallback


## Cell 7 - Validering av användarinmatning

`choose_option()` visar en numrerad lista och fortsätter fråga tills användaren gör ett giltigt val.

`read_positive_int()` använder `try/except` för att kontrollera att användaren skriver ett positivt heltal.

`select_booking_date()` skapar valbara datum för de kommande 14 dagarna med hjälp av `datetime` och `timedelta`.

`select_booking_time()` låter användaren välja mellan restaurangens fasta bokningstider.

In [ ]:
def choose_option(title, options):
    print(f"\n{title}")
    for number, option in enumerate(options, start=1):
        print(f"{number}. {option}")

    while True:
        try:
            choice = int(input("Choose an option: "))
            if 1 <= choice <= len(options):
                return choice - 1
            print(f"Choose a number between 1 and {len(options)}.")
        except ValueError:
            print("Please enter a number.")


def read_positive_int(message):
    while True:
        try:
            value = int(input(message))
            if value > 0:
                return value
            print("The number must be greater than zero.")
        except ValueError:
            print("Please enter a whole number.")


def select_booking_date(days_to_show=14):
    dates = [
        datetime.now().date() + timedelta(days=day)
        for day in range(days_to_show)
    ]
    labels = [date.strftime("%A, %Y-%m-%d") for date in dates]
    selected_index = choose_option("Available dates", labels)
    return dates[selected_index].strftime("%Y-%m-%d")


def select_booking_time():
    selected_index = choose_option("Available times", TIME_SLOTS)
    return TIME_SLOTS[selected_index]


## Cell 8 - Bokningsbekräftelse

`display_booking_confirmation()` visar information efter att en bokning har skapats.

Bekräftelsen innehåller boknings-ID, kundnamn, datum, tid, antal gäster och valt bord. Programmet visar även hur stor del av bordets kapacitet som används.

Boknings-ID:t är viktigt eftersom kunden behöver det för att kunna avboka bokningen.

In [ ]:
def display_booking_confirmation(booking, table):
    capacity_usage = (
        booking["guest_count"] / table.table_capacity
    ) * 100.0

    print("\n================================")
    print("      BOOKING CONFIRMED")
    print("================================")
    print(f"Booking ID: {booking['booking_id']}")
    print(f"Customer: {booking['customer_name']}")
    print(f"Date: {booking['booking_date']}")
    print(f"Time: {booking['booking_time']}")
    print(f"Guests: {booking['guest_count']}")
    print(f"Table: {booking['table_id']}")
    print(f"Capacity used: {capacity_usage:.1f}%")
    print("Save the booking ID. It is required for cancellation.")
    print("================================")


## Cell 9 - Bokningsflödet

`booking_flow()` kopplar samman programmets bokningsfunktioner.

Användaren väljer antal gäster, datum och tid. Programmet kontrollerar att tiden inte har passerat och söker därefter efter passande lediga bord.

Användaren får en rekommendation, ser information om de lediga borden och väljer sedan vilket bord som ska bokas. Bokningen sparas först efter att alla kontroller har genomförts.

In [ ]:
def booking_flow(tables, bookings):
    guest_count = read_positive_int("\nNumber of guests: ")
    booking_date = select_booking_date()
    booking_time = select_booking_time()

    selected_datetime = datetime.strptime(
        f"{booking_date} {booking_time}", "%Y-%m-%d %H:%M"
    )
    if selected_datetime <= datetime.now():
        print("The selected booking time has already passed.")
        return

    available_tables = find_available_tables(
        tables, bookings, guest_count, booking_date, booking_time
    )
    if not available_tables:
        print("No suitable tables are available at that time.")
        return

    preference = input(
        "Preferred table (window, quiet area, terrace, or anything): "
    ).strip()
    recommendation = recommend_table_with_grok(
        available_tables, preference or "No special preference"
    )
    print(f"\nRecommendation:\n{recommendation}")

    print("\nAvailable tables:")
    for table in available_tables:
        print(table.display_information())

    table_id = read_positive_int("\nTable ID to book: ")
    selected_table = next(
        (
            table
            for table in available_tables
            if table.table_id == table_id
        ),
        None,
    )
    if selected_table is None:
        print("The selected table is not available.")
        return

    customer_name = input("Name for the booking: ").strip()
    if not customer_name:
        print("The customer name cannot be empty.")
        return

    booking = create_booking(
        bookings,
        selected_table,
        customer_name,
        booking_date,
        booking_time,
        guest_count,
        BOOKINGS_FILE,
    )
    if booking is not None:
        display_booking_confirmation(booking, selected_table)


## Cell 10 - Huvudmeny och start

`main_menu()` laddar bord och bokningar när programmet startar. En `while`-loop håller menyn igång tills användaren väljer att avsluta.

Menyn innehåller tre alternativ:

1. Boka bord
2. Avboka bokning
3. Avsluta

Programmet startas genom att anropa `main_menu()`.

In [ ]:
def main_menu():
    tables = load_tables(TABLES_FILE)
    bookings = load_bookings(BOOKINGS_FILE)

    if not tables:
        print("No tables could be loaded.")
        return

    while True:
        choice = choose_option(
            "=== TABLEMATE ===",
            ["Book a table", "Cancel a booking", "Exit"],
        )

        if choice == 0:
            booking_flow(tables, bookings)
            input("\nPress Enter to return to the menu...")
        elif choice == 1:
            booking_id = read_positive_int("Booking ID to cancel: ")
            cancel_booking(bookings, booking_id, BOOKINGS_FILE)
            input("\nPress Enter to return to the menu...")
        else:
            print("Thank you for using TableMate.")
            break


In [ ]:
main_menu()

## Reflektion

Det mest utmanande var att koppla samman CSV-filer, objekt och användarinmatning. Jag lärde mig att CSV-värden först läses som text och därför måste omvandlas till rätt datatyper.

Jag valde CSV eftersom projektets data är liten och tabellformad. I en större version skulle jag använda en databas eftersom CSV inte är lämpligt när många användare bokar samtidigt.

En styrka är att bokningssystemet inte är beroende av att Grok fungerar. Python kontrollerar tillgängligheten och en lokal reservfunktion används om API-anropet misslyckas.

Projektet visar Python-syntax, funktioner, loopar, villkor, felhantering, objektorientering, arv, filhantering, standardbibliotek och ett externt bibliotek.